# Figure 3.6 experiment with $r=13/14$

Repeat the fixed-sample joint $(N,\lambda_N)$ experiment with

$$h_i=\left[0.01^{13/14}+\frac{i-1}{N-1}\left(2^{13/14}-0.01^{13/14}\right)\right]^{14/13}.$$

The sample and all other settings match the preceding linear-grid run: 200 standard-normal draws from `default_rng(20260823)`, bandwidth interval $[0.01,2]$, $N=100,200,\ldots,3000$, and $\lambda_N\in\{1,N^{1/2},N,N^{3/2},N^2,N^3\}$. The grid helper uses $\beta=2(1-r)=1/7$; the divergence remains one half of the squared $L^2$ distance.

Sources: [thesis grid definition, Eq. `eq:def_r_linear`](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [original Figure 3.6 workflow](experiment_1_joint_N_lambda_limit.ipynb); [grid helper](../kde_tools/bandwidths.py); [preceding linear-grid run](linear_grid_joint_N_lambda_limit.ipynb).

## Setup

Enable the shared automatic PDF exporter. Every displayed figure is saved under `../outputs/pdfs/r13over14_joint_n_lambda_limit/`. Data, sample, checkpoints, reference calculations, and validation diagnostics are stored separately under `../outputs/data/r13over14_joint_n_lambda_limit/`. The earlier r=1 results are preserved.

In [ ]:
%matplotlib inline

from pathlib import Path
import runpy
import sys

KDE_DIR = next(
    candidate.resolve()
    for ancestor in (Path.cwd(), *Path.cwd().parents)
    for candidate in (ancestor, ancestor / "Code" / "Code_kde")
    if (candidate / "kde_tools" / "pdf_export.py").is_file()
)
if str(KDE_DIR) not in sys.path:
    sys.path.insert(0, str(KDE_DIR))

from kde_tools.pdf_export import enable_automatic_pdf_saving
from threadpoolctl import threadpool_limits

NOTEBOOK_NAME = "r13over14_joint_N_lambda_limit"
pdf_directory = enable_automatic_pdf_saving(NOTEBOOK_NAME)
workflow = runpy.run_path(str(
    KDE_DIR / "outputs/analysis/linear_grid_joint_n_lambda_limit/run_experiment.py"
))
workflow["configure_grid"]("13/14")


## Run and validate

The reference is $\arg\min_{h\in[0.01,2]}\widehat Q_{13/14}(h)$, where $\widehat Q_{13/14}(h)=\frac12 h^{1/7}\|\partial_h\widehat f_h\|_2^2$. The calculation compares both endpoints and every local minimum detected on a 2001-point logarithmic scan, after refinement. Numerical checks use the exact Gaussian divergence on a 25-point grid and doubled frequency resolution for all six regimes at $N=3000$. Eigenpair residuals and spectral gaps are stored for every solve.

Sources: [thesis Proposition `prop:main_thm`](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [shared experiment and validation workflow](../outputs/analysis/linear_grid_joint_n_lambda_limit/run_experiment.py).

In [ ]:
with threadpool_limits(limits=2):
    results, reference, metric_h, metric_q = workflow["run_experiment"]()

print("Reference on [0.01, 2]:", reference)
display(results.pivot(index="N", columns="alpha", values="selected_h"))


## Result from the saved run

| lambda_N | Selected h at N=3000 |
| --- | ---: |
| 1 | 0.95654756 |
| N^(1/2) | 1.01551768 |
| N | 1.69448623 |
| N^(3/2) | 1.91244681 |
| N^2 | 1.97577039 |
| N^3 | 1.99857415 |

The numerical reference minimum on [0.01,2] is at the upper endpoint h=2, with Q_(13/14)=0.00853829185325. The detected interior local minimum is near h=0.56148648, where the criterion is larger (0.019292107181). Both endpoints and the interior minimum are compared explicitly.

![r=13/14 result](../outputs/images/r13over14_joint_n_lambda_limit/r13over14_joint_N_lambda_limit.png)

[Full numerical results](../outputs/data/r13over14_joint_n_lambda_limit/results.csv), [reference calculation](../outputs/data/r13over14_joint_n_lambda_limit/reference.json), and [validation](../outputs/data/r13over14_joint_n_lambda_limit/validation.json).

This describes the fixed sample and interval used here; the plot alone does not establish a convergence theorem. Run the next cell to regenerate the image and automatic PDF export.


In [ ]:
image_path = workflow["plot_results"](results, reference)
print("PNG:", image_path)
print("Automatic PDFs:", pdf_directory)
